<a href="https://colab.research.google.com/github/NataliaPoluektova/MyFirstProject/blob/main/Lab3_CMSI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Лабораторна робота №3.** Клієнтські та розподілені обчислення

**Мета роботи:** Ознайомитися з принципами розгортання моделей ШІ на кінцевих пристроях ($Edge\text{ }AI$); опанувати клієнтські обчислення в браузері через TensorFlow.js всередині Colab; вивчити особливості низькорівневої оптимізації пам'яті для мікроконтролерів ($TinyML$); спроектувати топологію розподіленої Edge-to-Cloud системи

**Блок 1. Клієнтські обчислення в браузері**


**Теоретичний базис:** Як працює ШІ всередині браузера

Коли ми запускаємо модель у хмарі, клієнт (браузер) є просто "німим" терміналом, який відправляє запит і чекає на відповідь.

У системі Web-AI браузер стає повноцінним обчислювальним вузлом.Це забезпечується трьоBenchmark-компонентами архітектури:

**1. Ізольоване середовище (Пісочниця JavaScript) **

Модель завантажується як статичний файл (набір JSON-структури графа та бінарних ваг) безпосередньо в оперативну пам'ять веб-вкладки. Обчислення відбуваються локально.

Це дає три залізні переваги:

100% Приватність (Privacy): Дані користувача (наприклад, зображення з веб-камери чи медичні показники) не відправляються на сторонні сервери. Вони обробляються та знищуються всередині сесії браузера.

Нульова затримка мережі (Zero Network Latency): Інференс не залежить від швидкості 4G/5G чи завантаженості серверів. Час відклику становить лічені мілісекунди.

Економіка інфраструктури (Cloud Cost Reduction): Обчислювальний рахунок за електрику та роботу процесорів оплачує сам користувач (його залізо), а не власник ШІ-сервісу.

**2. Апаратні бекенди (Hardware Backends) **

Сам по собі JavaScript є однопотоковим і надто повільним для важкої матричної математики (множення тензорів). Тому фреймворки типу TensorFlow.js використовують низькорівневі API для доступу до відеокарти ($GPU$) користувача:

CPU Backend: Найдешевший та найповільніший варіант. Використовує звичайний процесор. Для оптимізації застосовуються інструкції WASM (WebAssembly), які компілюють С/C++ код у швидкий бінарний формат для браузера, задіюючи векторні інструкції SIMD.

WebGL Backend: Класичний індустріальний стандарт. Фреймворк "обманює" відеокарту: він кодує матриці чисел (тензори) як пікселі текстур, а математичні операції нейромережі (згортки, активації) виконує за допомогою графічних шейдерів.

WebGPU Backend: Сучасний патерн (доступний у нових версіях Chrome/Edge). Це прямий спадкоємець Vulkan, Metal та DirectX 12 для вебу. Він прибирає графічний оверхед WebGL, дає прямий доступ до загальної пам'яті GPU, дозволяє використовувати обчислювальні шейдери ($Compute\text{ }Shaders$) та розкриває потенціал сучасних відеокарт майже на 100% від нативної швидкості.

**Крок 1. ** Тепер, розуміючи фізику процесу, ми можемо виконати інтерактивний запуск. Скопіюйте цей код у клітинку вашого Colab-ноутбука.

Завдяки магії команди %%html, Colab виділить ізольований IFrame, де завантажиться бібліотека TensorFlow.js і виконає навчання та інференс моделі прямо на вашій поточній відеокарті.

In [ ]:
%%html
<div style="background: #202124; color: #fff; padding: 20px; border-radius: 8px; font-family: sans-serif; max-width: 650px;">
    <h3 style="color: #34a853; margin-top: 0;">🤖 Клієнтський ШІ (Браузерний інференс)</h3>
    <p>Натисніть кнопку, щоб ініціалізувати обчислювальний граф у пам'яті вашого браузера:</p>
    <button id="start-btn" style="background: #34a853; color: white; border: none; padding: 10px 20px; border-radius: 4px; cursor: pointer; font-weight: bold; transition: 0.3s;">Запустити Web-AI</button>
    <div id="web-log" style="margin-top: 15px; background: #151515; color: #00ff00; padding: 15px; border-radius: 4px; font-family: monospace; font-size: 13px; min-height: 120px; white-space: pre-wrap; border: 1px solid #333;">Очікування команди від інженера...</div>
</div>

<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@4.20.0/dist/tf.min.js"></script>

<script>
document.getElementById('start-btn').addEventListener('click', async () => {
    const logZone = document.getElementById('web-log');
    logZone.style.color = "#00ff00";
    logZone.innerText = "⏳ Завантаження TensorFlow.js та аналіз архітектури заліза...\n";

    // 1. Очікуємо готовності інфраструктури та вибираємо найкращий доступний прискорювач
    await tf.ready();
    const activeBackend = tf.getBackend();
    logZone.innerText += `🚀 Апаратний прискорювач активовано: ${activeBackend.toUpperCase()}\n`;

    // 2. Створюємо лінійні тензори (Вхідні дані для функції y = 2x + 5)
    // Масиви автоматично завантажуються у VRAM відеокарти, якщо обрано WebGL/WebGPU
    const xs = tf.tensor1d([-1, 0, 1, 2, 3, 4]);
    const ys = tf.tensor1d([ 3, 5, 7, 9, 11, 13]);

    // 3. Конструюємо архітектуру обчислювального графа (1 шар, 1 нейрон)
    const model = tf.sequential();
    model.add(tf.layers.dense({units: 1, inputShape: [1]}));

    // 4. Компіляція графа з оптимізатором SGD (Stochastic Gradient Descent)
    model.compile({
        loss: 'meanSquaredError',
        optimizer: tf.train.sgd(0.05)
    });

    logZone.innerText += "🏋️‍♂️ Градієнтний спуск запущено локально на GPU клієнта (60 епох)...\n";

    // 5. Локальне навчання моделі
    await model.fit(xs, ys, {
        epochs: 60,
        callbacks: {
            onEpochEnd: (epoch, logs) => {
                if ((epoch + 1) % 20 === 0) {
                    logZone.innerText += `  └─ Епоха ${epoch + 1}: Помилка моделі (Loss) = ${logs.loss.toFixed(4)}\n`;
                }
            }
        }
    });

    // 6. Фінальний інференс: робимо прогноз для нового значення x = 10.
    // Очікувана математика: у = 2 * 10 + 5 = 25
    const testTensor = tf.tensor1d([10]);
    const prediction = model.predict(testTensor);

    // Забираємо результат з пам'яті GPU назад у JavaScript потік
    const result = await prediction.data();

    logZone.innerText += `\n🏁 ЛОКАЛЬНИЙ ІНФЕРЕНС ЗАВЕРШЕНО:\n`;
    logZone.innerText += `  └─ Вхідне значення x = 10 ──► Прогноз моделі y = ${result[0].toFixed(4)} (Очікувалось ~25.0)`;

    // КРИТИЧНО ДЛЯ WEB-AI: Ручне очищення виділеної пам'яті (VRAM)
    // У JavaScript Garbage Collector не вміє автоматично чистити пам'ять відеокарти!
    xs.dispose();
    ys.dispose();
    testTensor.dispose();
    prediction.dispose();
    model.dispose();
});
</script>

Коли ви натиснете кнопку, зверніть увагу на рядок Апаратний прискорювач активовано. Залежно від вашого браузера та наявності дискретної/інтегрованої відеокарти, там з'явиться WEBGL, WEBGPU або CPU (WASM). Це наочно демонструє, як абстрактний код адаптується під поточне залізо кінцевого користувача.

**Завдання для самостійного виконання 1**
 Дослідження масштабованості моделі та адаптація гіперпараметрів.Інженерна задача:Уявіть, що фізичний датчик на виробництві змінив режим роботи, і тепер його показники описуються іншою лінійною залежністю:$$y = 3x - 1$$

 Вам необхідно змінити вхідні дані, збільшити час навчання і перевірити модель на новому тестовому значенні.

 Покроковий алгоритм для студента:Оновлення даних (Тензорів):У тексті скрипта знайдіть рядки створення тензорів xs та ys. Замініть значення в ys так, щоб вони відповідали новій формулі

 $y = 3x - 1$:

 Замість минулих значень вписуємо нові:

const xs = tf.tensor1d([-1, 0, 1, 2, 3, 4]);

const ys = tf.tensor1d([-4, -1, 2, 5, 8, 11]);

Збільшення епох навчання:

Щоб модель краще підлаштувалася під новий нахил лінії, збільште кількість епох навчання у функції model.fit з 60 до 100.

Зміна тестового інференсу:

Змініть значення у testTensor для перевірки прогнозу з 10 на 5.(За новою формулою очікуваний результат: $3 \cdot 5 - 1 = 14$).

In [ ]:
#Ваш код:


**Блок 2. Обмеження RAM та концепція Tensor Arena**

На відміну від серверів чи смартфонів, де доступна динамічна віртуальна пам'ять, мікроконтролер (наприклад, архітектури ARM Cortex-M) має на борту всього від 16 до 512 Кілобайт SRAM (оперативної пам'яті).

У таких умовах стандартні підходи програмування PyTorch або TensorFlow стають руйнівними через дві головні проблеми:

1. Небезпека динамічної алокації (Купа / Heap)

У звичайному Python-коді під час виконання прямого проходу ($Forward\text{ }Pass$) фреймворк постійно виділяє нову пам'ять під проміжні матриці активацій і видаляє її після обчислень через Garbage Collector.

На мікроконтролері функції типу malloc() або оператори new швидко призводять до фрагментації пам'яті. Пам'ять покривається дрібними "дірками". У певний момент для чергового великого тензора згортки не знайдеться цілісного безперервного шматка RAM, пристрій викине помилку Out of Memory і критична інфраструктура (наприклад, кардіостимулятор чи датчик гальм автомобіля) просто зупиниться.

2. Інженерне рішення: Тензорна Арена (Tensor Arena)

Щоб гарантувати 100% стабільність системи, архітектура TensorFlow Lite for Microcontrollers (TFLM) використовує концепцію детермінованого статичного виділення пам'яті.

Інженер на етапі компіляції прошивки створює Тензорну Арену — один монолітний статичний масив байтів (uint8_t tensor_arena[kArenaSize]).

Під час старту пристрою спеціальний планувальник (Allocator) один раз заходить у цей масив і розрізає його на чіткі фіксовані блоки під кожен шар нейромережі.Пам'ять використовується повторно: як тільки ШІ порахував Шар 1, цей самий шматок Арени негайно віддається під збереження результатів Шару 2.

Якщо весь обчислювальний граф разом із вхідними буферами поміщається в Арену — модель працюватиме роками без жодного ризику падіння через брак RAM.

Математика оптимізації:

Вплив типів даних (Float32 vs INT8)Об'єм Тензорної Арени, який потрібно виділити в RAM, напряму залежить від розрядності чисел, якими оперує нейромережа.

Еталонний формат ($Float32$): Кожне значення активації або ваги записується як число з плаваючою крапкою розміром 4 байти (32 біти). Це забезпечує максимальну точність, але швидко виїдає дефіцитну RAM.

Квантований формат ($INT8$): За допомогою процедури квантування ($Quantization$) математичний простір чисел $[-\text{max}, +\text{max}]$ стискається в простір цілих чисел від $-128$ до $127$. Кожне значення тепер займає всього 1 байт (8 бітів).

Інженерний ефект: Перехід на INT8 зменшує вимоги до розміру статичної Тензорної Арени (RAM) та ваги моделі на диску (Flash) рівно в 4 рази, дозволяючи запускати згорткові шари на чіпах вартістю в пару доларів.

**Крок 2. **Симуляція обмежень пам'яті TinyML


Цей блок виконує математичний аудит пам'яті для мікроконтролера. Ми розраховуємо розмір статичної Тензорної Арени (Tensor Arena), яка є обов'язковою для архітектури TensorFlow Lite for Microcontrollers (TFLM).

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# 1. Створюємо прототип графа нейромережі для обробки сигналів з IoT-сенсора
class TinyMLAudioNet(nn.Module):
    def __init__(self):
        super(TinyMLAudioNet, self).__init__()
        # Вхід: 40 ознак спектрограми звуку (наприклад, з мікрофона)
        self.fc1 = nn.Linear(40, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 4) # 4 класи звукових подій

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

tiny_model = TinyMLAudioNet().eval()

# 2. Розрахунок пам'яті під активації (Тензорна Арена)
# На мікроконтролері розмір бачу завжди дорівнює 1
input_shape = (1, 40)
hidden_shape = (1, 16)
output_shape = (1, 4)

# Студенти аналізують різницю між типами даних Float32 (4 байти) та INT8 (1 байт)
bytes_float32 = 4
bytes_int8 = 1

# Розрахунок для сирого графа (Float32)
arena_float32 = (np.prod(input_shape) + np.prod(hidden_shape) + np.prod(output_shape)) * bytes_float32

# Розрахунок після оптимізації та квантування графа (INT8)
arena_int8 = (np.prod(input_shape) + np.prod(hidden_shape) + np.prod(output_shape)) * bytes_int8

print(" АУДИТ ОБЧИСЛЮВАЛЬНОЇ ПАМ'ЯТІ ДЛЯ TFLM:")
print(f"  ├─ Необхідна Tensor Arena (Базова модель Float32): {arena_float32} байтів")
print(f"  ├─ Необхідна Tensor Arena (Квантована модель INT8):   {arena_int8} байтів")
print(f"  └─  Економіка дефіцитної RAM мікроконтролера: Зменшено в {arena_float32/arena_int8:.1f} рази!")

# 3. Імітація генерації заголовного файлу прошивки C++
weights = tiny_model.fc1.weight.detach().numpy().flatten()[:5]
c_array = ", ".join([str(round(float(w), 4)) for w in weights])
print(f"\n Фрагмент масиву для вшивання у Flash-пам'ять мікроконтролера:")
print(f"static const float tiny_model_weights[] = {{ {c_array}, ... }};")

**Завдання для самостійного виконання 2**

Модифікуйте код у Клітинці 2. Припустімо, що прихований шар моделі збільшився з 16 нейронів до 64 (self.fc1 = nn.Linear(40, 64), self.fc2 = nn.Linear(64, 4)). Перерахуйте, яким буде новий мінімальний об'єм статичної Тензорної Арени для форматів Float32 та INT8.

In [ ]:
#Ваш код:


**Блок 3** Гібридні обчислення та Федеративне навчання

У промислових системах ШІ архітектура будується на гібридному конвеєрі Edge-to-Cloud (Від периферії до хмари). Це дозволяє гнучко розподіляти ресурси:

Edge-рівень (Кінцеві пристрої): Камери або датчики. Вони мають слабкі процесори, але стоять на місці збору даних. Тут виконується миттєвий локальний інференс ($Latency < 5\text{ мс}$).

Cloud-рівень (Центральна хмара): Потужні дата-центри. Тут збирається глобальна аналітика та виконується важке перенавчання моделей.

Коли дані з камер чи датчиків не можна передавати в хмару через закони про приватність або дорогий інтернет-трафік, застосовують патерн Федеративного навчання (Federated Learning).Замість того, щоб збирати приватні дані користувачів на одному сервері, хмара розсилає копію моделі на кінцеві пристрої. Пристрої донавчають її локально кожен на своїх даних і повертають назад у хмару не самі дані, а тільки змінені ваги (градієнти). Хмарний сервер за алгоритмом FedAvg (Federated Averaging) усереднює ці зміни, оновлює глобальну модель і знову розсилає її на пристрої.

**Крок 3. **Цей блок симулює роботу конвеєра: спочатку ми обробляємо безперервний потік даних з датчика «на льоту» за допомогою ковзного часового вікна (Edge), а потім імітуємо роботу алгоритму FedAvg у хмарі, об'єднуючи знання з двох різних заводів.

In [ ]:
import torch
import numpy as np

print(" Ініціалізація симулятора Edge-to-Cloud конвеєра...\n")

# 1. ЕМУЛЯЦІЯ EDGE-РІВНЯ: Обробка потоку даних через ковзне вікно
# Симулюємо безперервний потік чисел з датчика верстата
streaming_data = np.sin(np.linspace(0, 50, 100)) + np.random.normal(0, 0.1, 100)

window_size = 10  # Розмір ковзного вікна
edge_buffer = []
anomalies_detected = 0

# Обробляємо потік «в реальному часі»
for i in range(100):
    current_reading = streaming_data[i]
    edge_buffer.append(current_reading)

    if len(edge_buffer) > window_size:
        edge_buffer.pop(0) # Викидаємо найстаріше значення (зсув вікна)

        # Локальний інференс: якщо середнє значення у вікні перевищує поріг
        local_mean = np.mean(edge_buffer)
        if abs(local_mean) > 1.1:
            anomalies_detected += 1

print("⏱ 1. Результат потокової обробки на Edge-пристрої:")
print(f"  ├─ Оброблено точок потоку: 100")
print(f"  └─ Детектовано локальних аномалій: {anomalies_detected}\n")


# 2. ЕМУЛЯЦІЯ CLOUD-РІВНЯ: Федеративне усереднення ваг (FedAvg)
print("☁️ 2. Результат агрегації в Хмарі (Federated Learning):")

# Уявимо, що два різні заводи локально навчили свої моделі і прислали їхні ваги в хмару
weights_factory_A = torch.tensor([0.40, 0.10, -0.80])
weights_factory_B = torch.tensor([0.60, 0.20, -0.60])

print(f"  ├─ Локальні ваги від Заводу А: {weights_factory_A.tolist()}")
print(f"  ├─ Локальні ваги від Заводу Б: {weights_factory_B.tolist()}")

# Алгоритм FedAvg: хмарний сервер знаходить математичне середнє ваг
global_weights = (weights_factory_A + weights_factory_B) / 2

print(f"  └─  Оновлені ГЛОБАЛЬНІ ваги моделі: {global_weights.tolist()}")

**Самостійне завдання 3.** Стрес-тестування ковзного вікна та модифікація ваг федерації.

Покроковий алгоритм для студента:
Зміна параметрів Edge-вікна:
У коді симулятора знайдіть змінну window_size (розмір вікна). Зменште її розмір з 10 до 5. А поріг детекції аномалій (у рядку if abs(local_mean) > 1.1:) знизьте з 1.1 до 0.7. Перезапустіть клітинку.

Зміна даних федерації (Cloud):
Уявіть, що Завод Б оновив своє обладнання, і його модель передала інші ваги. Змініть координати тензора weights_factory_B на: [0.80, 0.30, -0.40]. Перезапустіть обчислення.

In [ ]:
#Ваш код:

Ваші висновки:
